
# Medical Text Classification with Bidirectional LSTM and Attention

This notebook is the experiment-oriented companion to the modular project code.

> **Medical disclaimer:** Educational portfolio demonstration only. This is not a diagnostic tool. Do not use real patient identifiers, protected health information, or confidential clinical records.


In [ ]:

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import DEFAULT_SAMPLE_DATA, MODEL_DIR, OUTPUT_DIR, TrainingConfig
from src.data_preprocessing import load_and_prepare_dataset
from src.visualization import save_class_distribution, save_text_length_distribution



## 1. Dataset audit

The included file contains only ten synthetic rows—two for each of five specialties. It is intentionally safe for a public portfolio, but it cannot support credible model-performance claims.


In [ ]:

dataframe, audit = load_and_prepare_dataset(
    DEFAULT_SAMPLE_DATA,
    minimum_class_count=2,
    preprocessing_mode="legacy",  # matches the supplied demonstration model
)
display(dataframe.head())
audit.to_dict()


In [ ]:

dataframe["label"].value_counts()



## 2. Preprocessing choices

The project provides two preprocessing modes:

- `legacy`: exactly matches the supplied model artifact.
- `clinical_safe`: recommended for retraining because it preserves numeric and punctuation context such as `140/90`, `98%`, and `2.5-mg`.

Negations are not removed.


In [ ]:

from src.medical_text_preprocessing import clean_medical_text

example = "<b>No fever</b>; BP 140/90, O2 98%, dose 2.5-mg."
print("legacy      :", clean_medical_text(example, mode="legacy"))
print("clinical_safe:", clean_medical_text(example, mode="clinical_safe"))



## 3. Exploratory outputs


In [ ]:

save_class_distribution(dataframe, OUTPUT_DIR / "class_distribution.png")
save_text_length_distribution(dataframe, OUTPUT_DIR / "text_length_distribution.png")



## 4. Classical NLP baseline

A TF-IDF + Logistic Regression baseline is included to verify that the project compares deep learning against a simpler approach. With the ten-row sample, this is only a pipeline diagnostic.


In [ ]:

from src.baseline import evaluate_tfidf_logistic_baseline

baseline_metrics, baseline_analysis = evaluate_tfidf_logistic_baseline(dataframe)
baseline_metrics.to_record()



## 5. Train a new BiLSTM + Attention model

Run this section only after installing TensorFlow. Replace the ten-row sample with an appropriately licensed, de-identified, representative dataset before using the resulting metrics in a portfolio.


In [ ]:

# Uncomment to train.
# from src.model_training import train_model
#
# config = TrainingConfig(
#     epochs=20,
#     batch_size=32,
#     preprocessing_mode="clinical_safe",
# )
# training_result = train_model(
#     dataframe,
#     model_directory=MODEL_DIR,
#     output_directory=OUTPUT_DIR,
#     config=config,
# )



## 6. Load the supplied demonstration artifact


In [ ]:

# Requires TensorFlow.
# from src.inference_pipeline import MedicalTextInferencePipeline
#
# pipeline = MedicalTextInferencePipeline().load()
# result = pipeline.predict(
#     "Patient reports chest discomfort, palpitations, and hypertension."
# )
# result



## 7. Honest interpretation of the supplied model

The original notebook used 10 rows, producing a 5-row training set, 2-row validation set, and 3-row test set. The saved model obtained 33.3% accuracy and 0.10 macro F1 on that three-row holdout. It predicted `Orthopedic` for all three test rows with probabilities near 20% for every class. These numbers show that the code path runs; they do not show useful generalization.

The professional next step is to retrain on a substantially larger, appropriately licensed and de-identified dataset, preserve a leakage-safe split, and report macro F1, weighted F1, per-class recall, confusion patterns, and uncertainty.
